In [10]:
# CELL 3: StyleGAN2 Setup (Run once)

import subprocess
from pathlib import Path

# Define paths
PROJECT_ROOT = Path("..").resolve()
STYLEGAN2_DIR = PROJECT_ROOT / "stylegan2-ada-pytorch"
MODELS_DIR = PROJECT_ROOT / "models" / "pretrained"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("="*60)
print("STYLEGAN2 SETUP")
print("="*60)

# Clone StyleGAN2 if not exists
if not STYLEGAN2_DIR.exists():
    print("\n📥 Cloning StyleGAN2-ADA-PyTorch...")
    subprocess.run([
        "git", "clone", 
        "https://github.com/NVlabs/stylegan2-ada-pytorch.git",
        str(STYLEGAN2_DIR)
    ])
    print("✅ Cloned successfully!")
else:
    print("✅ StyleGAN2 already cloned")

# Add to Python path
import sys
if str(STYLEGAN2_DIR) not in sys.path:
    sys.path.insert(0, str(STYLEGAN2_DIR))
    print(f"✅ Added to sys.path: {STYLEGAN2_DIR}")

# Download pre-trained FFHQ model if not exists
FFHQ_MODEL = MODELS_DIR / "ffhq.pkl"
if not FFHQ_MODEL.exists():
    print("\n📥 Downloading pre-trained FFHQ model (takes a few minutes)...")
    import urllib.request
    url = "https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl"
    urllib.request.urlretrieve(url, str(FFHQ_MODEL))
    print(f"✅ Downloaded to: {FFHQ_MODEL}")
else:
    print(f"✅ FFHQ model exists: {FFHQ_MODEL}")

print("\n" + "="*60)
print("✅ StyleGAN2 setup complete!")
print("="*60)

HTTPError: HTTP Error 403: Forbidden

In [6]:
# CELL 4 (UPDATED): Prepare & Resize Caricature Dataset for StyleGAN2

import shutil
import subprocess
import sys
from pathlib import Path
from PIL import Image

# ----------- Paths -----------

PROJECT_ROOT = Path("..").resolve()
ORIGINAL_ROOT = PROJECT_ROOT / "data" / "raw" / "OriginalImages"
STYLEGAN2_DIR = PROJECT_ROOT / "stylegan2-ada-pytorch"
STYLEGAN_DATA_DIR = PROJECT_ROOT / "data" / "stylegan_training"

IMAGES_DIR = STYLEGAN_DATA_DIR / "images"
ZIP_PATH = STYLEGAN_DATA_DIR / "caricatures.zip"

STYLEGAN_DATA_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(exist_ok=True)

print("=" * 60)
print("PREPARING CARICATURE DATASET (RESIZE & CROP)")
print("=" * 60)

# ----------- Collect caricature images -----------

image_files = []
for person_dir in ORIGINAL_ROOT.iterdir():
    if person_dir.is_dir():
        for img in person_dir.iterdir():
            if (
                img.is_file()
                and img.suffix.lower() in {".png", ".jpg", ".jpeg"}
                and img.name.lower().startswith("c")
            ):
                image_files.append(img)

image_files = sorted(image_files)
print(f"Found {len(image_files)} caricature images")

assert image_files, "❌ No caricature images found!"

# ----------- Resize & center crop to 256x256 -----------

TARGET_SIZE = 256

for i, img_path in enumerate(image_files):
    with Image.open(img_path) as img:
        # convert to RGB
        img = img.convert("RGB")
        
        # center crop to square
        w, h = img.size
        min_dim = min(w, h)
        left = (w - min_dim) // 2
        top = (h - min_dim) // 2
        img = img.crop((left, top, left + min_dim, top + min_dim))
        
        # resize to 256x256
        img = img.resize((TARGET_SIZE, TARGET_SIZE), Image.LANCZOS)
        
        # save to stylegan images
        out_path = IMAGES_DIR / f"img{i:05d}.png"
        img.save(out_path)

print("✅ Images resized & saved to:", IMAGES_DIR)

# ----------- Remove old ZIP -----------

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
    print("🗑️ Removed old ZIP")

# ----------- Create dataset ZIP -----------

cmd = [
    sys.executable,
    str(STYLEGAN2_DIR / "dataset_tool.py"),
    "--source", str(IMAGES_DIR),
    "--dest", str(ZIP_PATH),
]

print("\n📦 Creating dataset.zip...")
result = subprocess.run(cmd, capture_output=True, text=True)

print("\nSTDOUT:\n", result.stdout)
print("\nSTDERR:\n", result.stderr)

assert ZIP_PATH.exists(), "❌ ZIP not created after dataset_tool"
print(f"\n✅ Dataset created successfully: {ZIP_PATH}")
print("=" * 60)

PREPARING CARICATURE DATASET (RESIZE & CROP)
Found 6042 caricature images
✅ Images resized & saved to: D:\Projects 2025\Caricature Generation\data\stylegan_training\images
🗑️ Removed old ZIP

📦 Creating dataset.zip...

STDOUT:
 

STDERR:
 
100%|##########| 6042/6042 [01:26<00:00, 69.58it/s]


✅ Dataset created successfully: D:\Projects 2025\Caricature Generation\data\stylegan_training\caricatures.zip


In [14]:
# Download MetFaces-256 pretrained StyleGAN2-ADA model

import urllib.request
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
PRETRAINED_DIR = PROJECT_ROOT / "models" / "pretrained"
PRETRAINED_DIR.mkdir(parents=True, exist_ok=True)

METFACES_256 = PRETRAINED_DIR / "metfaces256.pkl"

URL = "https://huggingface.co/datasets/valhalla/stylegan2-ada-pretrained/resolve/main/metfaces256.pkl"

if not METFACES_256.exists():
    print("📥 Downloading MetFaces-256 pretrained model...")
    urllib.request.urlretrieve(URL, METFACES_256)
    print("✅ Downloaded:", METFACES_256)
else:
    print("✅ MetFaces-256 already exists:", METFACES_256)


📥 Downloading MetFaces-256 pretrained model...


HTTPError: HTTP Error 401: Unauthorized

In [12]:
# CELL 5: Fine-tune StyleGAN2-ADA on Caricatures (FFHQ-256)

import subprocess
import sys
from pathlib import Path

# ---------------- Paths ----------------
PROJECT_ROOT = Path("..").resolve()
STYLEGAN2_DIR = PROJECT_ROOT / "stylegan2-ada-pytorch"
STYLEGAN_DATA_DIR = PROJECT_ROOT / "data" / "stylegan_training"
TRAINING_DIR = PROJECT_ROOT / "models" / "training_runs"

TRAINING_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("FINE-TUNING STYLEGAN2-ADA (FFHQ-256 → CARICATURES)")
print("=" * 60)

# ---------------- Training command ----------------
cmd = [
    sys.executable,
    str(STYLEGAN2_DIR / "train.py"),
    "--data", str(STYLEGAN_DATA_DIR / "caricatures.zip"),
    "--resume", "ffhq256",          # 🔥 built-in 256x256 pretrained model
    "--outdir", str(TRAINING_DIR),
    "--gpus", "1",
    "--kimg", "1000",
    "--aug", "ada"
]

print("\n🚀 Starting training with command:\n")
print(" ".join(cmd))

# ---------------- Run training ----------------
result = subprocess.run(
    cmd,
    cwd=str(STYLEGAN2_DIR)
)

print("\nTraining process finished with return code:", result.returncode)
print("=" * 60)

FINE-TUNING STYLEGAN2-ADA (FFHQ-256 → CARICATURES)

🚀 Starting training with command:

C:\Users\nahia\AppData\Local\Programs\Python\Python311\python.exe D:\Projects 2025\Caricature Generation\stylegan2-ada-pytorch\train.py --data D:\Projects 2025\Caricature Generation\data\stylegan_training\caricatures.zip --resume ffhq256 --outdir D:\Projects 2025\Caricature Generation\models\training_runs --gpus 1 --kimg 1000 --aug ada

Training process finished with return code: 1
